# verl.workers.megatron_workers

verl.workers.megatron_workers 是 verl 框架中专门用于对接 Megatron-LM 分布式训练后端的模块。

如果说 verl 是 RLHF 的“总指挥”，那么 megatron_workers 就是其中的“重型火炮手”。它负责在超大规模模型（如 70B+ 参数）场景下，利用 Megatron-LM 的 3D 并行能力（张量并行 TP、流水线并行 PP、数据并行 DP）以及 DeepSpeed 的 ZeRO 优化技术，来执行具体的模型训练和推理任务。

以下是该模块的架构设计、核心组件及工作机制的详细解读。
## 🏗️ 核心架构：基于 Ray 的分布式 Actor
<font color='red'>verl 并没有直接调用 Megatron 的脚本，而是将其封装为 Ray Actor。这意味着每一个 Megatron 的训练进程（Worker）都是一个 Ray 对象，可以被 Driver 进程远程调用。</font>

该模块主要包含以下几个核心类：

1. <b>MegatronWorker (基类)</b>
这是所有 Megatron 后端 Worker 的父类。
- 职责：负责初始化 Megatron 的核心环境（如 initialize_megatron）、设置分布式通信组（Process Groups）、以及管理模型在 GPU 间的切分逻辑。
- 关键机制：它处理了 Megatron 特有的“模型并行世界大小”与 Ray 分配的 GPU 资源之间的映射关系。

2. <b>MegatronActorRolloutRefWorker (核心组件)</b>
这是 RLHF 中最复杂的组件，它在一个 Worker 中集成了三个角色的功能（通常通过共置 Colocation 来节省显存）：
- Actor (策略网络)：负责更新策略。
- Rollout (采样)：负责生成数据。
- Ref (参考网络)：用于计算 KL 散度，防止模型跑偏。
- Megatron 特性：
    - 训练态：使用 Megatron 的 TP/PP 并行进行高效的梯度更新。
    - 推理态：利用 3D-HybridEngine 技术。在生成数据时，它会自动将模型权重从“训练分片状态”重分片（Resharding）为“推理状态”（通常是 TP 并行或 vLLM 格式），生成完后再切回训练状态。这避免了重新加载模型，极大提升了吞吐量。
3. <b>MegatronCriticWorker</b>
- 职责：负责训练 Critic 模型（价值网络），用于估算优势函数（Advantage）。
- 实现：同样基于 Megatron 后端，支持独立的 TP/PP 配置。

## ⚙️ 关键工作机制
### 1.<font color='green'> 3D 并行与 DeepSpeed ZeRO 的融合</font>
<font color='red'>在 megatron_workers 中，verl 并没有二选一，而是结合了两者：</font>
- <font color='red'>Megatron-LM：</font><font color='green'>负责计算层面的并行。</font>它处理张量并行（TP，切分矩阵乘法）和流水线并行（PP，切分模型层）。
- <font color='red'>DeepSpeed：</font><font color='green'>负责显存层面的优化。</font>Worker 内部会初始化 DeepSpeed 引擎（通常是 ZeRO-3 模式），对优化器状态和梯度进行分片。

- 结果：<font color='red'>verl 通过 megatron_workers 实现了 TP + PP + ZeRO-3 的极致显存节省策略，使得单卡显存能容纳更大的模型。</font>

### 2. 动态混合引擎 (3D-HybridEngine)
这是 verl 对原生 Megatron 的最大改造之一。
- 痛点：原生 Megatron 训练和推理的并行策略通常不同（训练用大 TP 或 PP，推理用小 TP）。传统做法需要在两个进程间拷贝权重，极慢。
- 解决方案：<font color='red'>MegatronActorRolloutRefWorker 实现了原地重分片。</font>
    - <font color='red'>在 训练阶段</font>，权重按照 Megatron 的 TP/PP 策略分布在 GPU 上。
    - <font color='red'>在 Rollout 阶段</font>，Worker 内部调用 all_gather 或重排操作，将权重重组为适合推理的格式（例如 TP=1 或 TP=2），利用 vLLM 或 Megatron 的推理接口生成数据，生成结束后立即释放并恢复训练分片。
    - 优势：<font color='red'>无需跨进程传输权重，显存零冗余。</font>
    
### 3. 配置与资源映射
在使用 megatron_workers 时，你需要通过配置文件（YAML）精确控制并行维度。
典型配置示例：


In [ ]:
actor_rollout_ref:
  actor:
    strategy: "megatron"  # 激活 megatron_workers
    megatron:
      tensor_model_parallel_size: 4   # TP=4
      pipeline_model_parallel_size: 2 # PP=2
      param_offload: true             # 开启 DeepSpeed 参数卸载到 CPU
      grad_offload: true              # 开启梯度卸载

## 📊 代码层面的交互流程
当你运行 verl 的 PPO 训练时，megatron_workers 的交互流程如下：
1. <font color='red'>初始化：</font>

    - Ray Driver 启动 MegatronActorRolloutRefWorker。Worker 内部调用 Megatron 的初始化函数，建立 TP/PP 通信组，并加载预训练模型权重。

2. <font color='red'>Rollout (采样)：</font>

    - Driver 调用 Worker 的 generate_sequences 方法。Worker 内部触发 HybridEngine 切换，重组权重，执行前向推理，返回 Prompt 的回复。

3. <font color='red'>Compute (计算)：</font>

    - Driver 调用 compute_log_probs 或 update_critic。Worker 利用 Megatron 的分布式能力进行大规模矩阵运算。

4. <font color='red'>Update (更新)：</font>

    - Driver 调用 update_actor。Worker 执行反向传播，利用 DeepSpeed 的 ZeRO-3 优化器更新参数，并进行梯度同步。

## 📌 总结

verl.workers.megatron_workers 是 verl 框架能够支持千亿参数模型 RLHF 训练的基石。
- 对于开发者：你不需要直接编写 Megatron 的底层代码，只需在配置文件中指定 strategy: megatron 并设置好 TP/PP 参数。
- 对于系统：它屏蔽了复杂的分布式通信细节，提供了统一的 Worker 接口，让 RLHF 算法（PPO/GRPO）可以像操作单机模型一样操作分布式大模型。

# verl/workers/megatron_workers/ActorRolloutRefWorker

<font color='red'>VeRL 如何利用 Ray、PyTorch 分布式和 Megatron-LM 来构建一个高性能的 RLHF（强化学习人类反馈）训练环境。</font>

class ActorRolloutRefWorker(MegatronWorker, DistProfilerExtension):
- <font color='red'>\_\_init\_\_()</font>
- _build_model_optimizer()
- _build_rollout()
- init_model()
- update_actor()
- generate_sequences()
- compute_ref_log_prob()
- compute_log_prob()
- load_checkpoint()
- load_pretrained_model()
- save_checkpoint()

## __init__()
- 这段代码的核心逻辑是：<font color='red'>基于配置文件，初始化分布式环境，并根据 role 参数决定该 Worker 实例承担的具体任务（Actor、Rollout 或 Ref）</font>

In [ ]:
from omegaconf import DictConfig, OmegaConf  
from verl.single_controller.base import Worker
from megatron.core import parallel_state as mpu

    """
    This worker can be instantiated as a standalone actor or a standalone rollout or a standalone reference policy
    or a hybrid engine based on the config.rollout
    """

    def __init__(self, config: DictConfig, role: str, **kwargs):
        Worker.__init__(self)
        self.config = config
        if repatch is not None:
            # NPU MindSpeed patch, will be refactored with MindSpeedEngine.
            repatch(self.config.actor.megatron.get("override_transformer_config", {}))

        # NOTE(sgm): We utilize colocate WorkerGroup by default.
        # As a result, Workers for different model share the same process.
        # Therefore, we only require one distribute initialization.
        # To utilize different parallel strategy in different models:
        # 1, users should disable WorkerDict; 
        # 2.assign different ResourcePool to different models,
        
        '''
        注释提及的 Ray Patch：代码中的注释提到了 ray==2.10 的一个 Patch。
        这是为了解决 Ray 在同址部署（Colocate）场景下，
        不同模型组之间资源分配不均的问题（即如何让不同的模型使用不同的并行策略或资源池）。
        '''
        # 3. and apply the following patch in ray==2.10, https://github.com/ray-project/ray/pull/44385
        
        '''
        1.分布式环境的单例初始化:
        
        核心逻辑：VeRL 使用了 WorkerGroup Colocate（同址部署） 策略。
        这意味着同一个物理进程（GPU）上可能同时运行 Actor、Rollout 和 Ref 模型。
        
        作用：这段代码确保了即使一个进程里有多个 Worker 实例（例如同时有 Actor 和 Ref），
        也只会初始化一次 NCCL 通信后端和并行策略（Tensor Parallelism, Pipeline Parallelism 等），
        避免了资源冲突。
        
        
        '''
        if not torch.distributed.is_initialized():
            rank = int(os.environ["LOCAL_RANK"])
            torch.distributed.init_process_group(
                backend=get_nccl_backend(),
                timeout=datetime.timedelta(seconds=self.config.get("nccl_timeout", 600)),
                init_method=os.environ.get("DIST_INIT_METHOD", None),
            )
            get_torch_device().set_device(rank)

            mpu.initialize_model_parallel(
                tensor_model_parallel_size=self.config.actor.megatron.tensor_model_parallel_size,
                pipeline_model_parallel_size=self.config.actor.megatron.pipeline_model_parallel_size,
                virtual_pipeline_model_parallel_size=self.config.actor.megatron.virtual_pipeline_model_parallel_size,
                pipeline_model_parallel_split_rank=None,
                use_sharp=False,
                context_parallel_size=self.config.actor.megatron.context_parallel_size,
                expert_model_parallel_size=self.config.actor.megatron.expert_model_parallel_size,
                expert_tensor_parallel_size=self.config.actor.megatron.expert_tensor_parallel_size,
                nccl_communicator_config_path=None,
            )

        is_collect = (
            mpu.get_tensor_model_parallel_rank() == 0
            and mpu.get_pipeline_model_parallel_rank() == mpu.get_pipeline_model_parallel_world_size() - 1
            and mpu.get_context_parallel_rank() == 0
        )
        self._register_dispatch_collect_info(
            mesh_name="actor", dp_rank=mpu.get_data_parallel_rank(), is_collect=is_collect
        )

        set_random_seed(seed=self.config.actor.megatron.seed)
        
        '''
        2.通过 role 参数，Worker 可以扮演不同的角色。:
        - Standalone（独立模式）：如 "actor"，仅代表自己。
        - Hybrid（混合模式）：如 "actor_rollout_ref"，
          代表一个 Worker 实例同时承担采样、推演和参考模型的职责（极大节省显存，因为不需要为 Ref 模型单独存一份参数）。
        '''

        self.role = role
        assert self.role in ["actor", "rollout", "ref", "actor_rollout", "actor_rollout_ref"]

        '''
        通过 if self._is_actor: 来判断是否需要执行特定的逻辑。
        '''
        self._is_actor = self.role in ["actor", "actor_rollout", "actor_rollout_ref"]
        self._is_rollout = self.role in ["rollout", "actor_rollout", "actor_rollout_ref"]
        self._is_ref = self.role in ["ref", "actor_rollout_ref"]

        profiler_config = omega_conf_to_dataclass(config.get("profiler"))
        DistProfilerExtension.__init__(self, DistProfiler(rank=self.rank, config=profiler_config))

        # TODO(sgm): Currently, we only support reference model param offload
        # will support other offload later
        self._is_offload_param = False
        self._is_offload_grad = False
        self._is_offload_optimizer = False

        '''
        3. 配置归一化与 Offload 支持:
        
        
        '''
        # normalize config
        if self._is_actor and self._is_rollout:
            # 调整 Batch Size
            '''
            Actor 和 Rollout 混合时，需要根据并行度（Data Parallel Size）
            和 Rollout 的采样次数调整 Batch Size，以保证全局 Batch Size 一致。
            
            '''
            self.config.actor.ppo_mini_batch_size *= self.config.rollout.n
            self.config.actor.ppo_mini_batch_size //= mpu.get_data_parallel_world_size()
            if self.config.actor.get("ppo_micro_batch_size", None):
                self.config.actor.ppo_micro_batch_size //= mpu.get_data_parallel_world_size()
                self.config.rollout.log_prob_micro_batch_size //= mpu.get_data_parallel_world_size()
                self.config.actor.ppo_micro_batch_size_per_gpu = self.config.actor.ppo_micro_batch_size
                self.config.rollout.log_prob_micro_batch_size_per_gpu = self.config.rollout.log_prob_micro_batch_size

            '''
            卸载（Offload）配置:
            '''
            self._is_offload_param = self.config.actor.megatron.get("param_offload", False)
            self._is_offload_grad = self.config.actor.megatron.get("grad_offload", False)
            self._is_offload_optimizer = self.config.actor.megatron.get("optimizer_offload", False)
        elif self._is_ref:
            if self.config.ref.get("log_prob_micro_batch_size", None):
                self.config.ref.log_prob_micro_batch_size //= mpu.get_data_parallel_world_size()
                self.config.ref.log_prob_micro_batch_size_per_gpu = self.config.ref.log_prob_micro_batch_size
            else:
                assert self.config.ref.get("log_prob_micro_batch_size_per_gpu", None) is not None, (
                    "Please note that in the ref policy configuration, `log_prob_micro_batch_size_per_gpu` and "
                    "`log_prob_micro_batch_size` should not be None at the same time."
                )
            self._ref_is_offload_param = self.config.ref.megatron.get("param_offload", False)


## mpu.initialize_model_parallel(...)

- 在大规模语言模型（LLM）训练中，单张 GPU 的显存和算力远远无法满足需求。因此，<font color='red'>Megatron 引入了多种并行策略，将模型和数据拆分到成百上千张 GPU 上协同工作。</font>
- mpu.initialize_model_parallel(...) 的作用就是<font color='red'>根据你传入的配置，在底层的 PyTorch 分布式环境之上，构建出这些复杂的通信组（Process Groups）。</font>
- mpu 是 Model Parallelism Utilities（模型并行工具集） 的缩写。

### 🧩 核心并行维度解析
为了让你更直观地理解，我们可以把这段代码中的参数拆解为几个核心维度：
#### 1. 张量并行 (Tensor Parallelism, TP)-切分tensor
- 对应参数：<font color='red'>tensor</font>_model_parallel_size
- 通俗解释：<font color='green'>“把一层神经网络横向切开”。</font>
    - 比如一个巨大的矩阵乘法，TP 会把它拆分到多张 GPU 上同时计算，最后再把结果拼起来。这主要利用了 GPU 之间极高的互联带宽（如 NVLink）。
    - 它相当于把每一块单独的拼图都切成两半，分给两个人同时去拼。两个人必须紧密配合，每拼好一小块就要立刻对接一下，最后才能拼出完整的图案。
    - 在深度学习中，模型的核心计算就是矩阵乘法（例如 Y = X × W，其中 X 是输入数据，W 是模型的权重参数）。TP 就是利用矩阵运算的数学特性，把这个巨大的计算任务切分开。
    - 多头注意力（Multi-Head Attention）：比如模型有 32 个注意力头（Attention Heads），开启 TP=4 的张量并行后，每张显卡就只负责计算其中的 8 个头。大家各自算完，再把结果拼起来。
    - 前馈神经网络（FFN / MLP）：Transformer 里的两层巨大线性层，第一层通常采用列切分，第二层采用行切分。这样一拆一合，既分摊了显存压力，又保证了计算结果的正确性。
- 代码中的体现：self.config.actor.megatron.tensor_model_parallel_size 决定了切分模型张量需要几张卡。

#### 2. 流水线并行 (Pipeline Parallelism, PP)
- 对应参数：<font color='red'>pipeline</font>_model_parallel_size
- 通俗解释：“把模型按层纵向切开”。
    - 比如<font color='red'>一个 100 层的 Transformer</font>，PP=4 意味着前 25 层放在 GPU 1，中间 25 层放在 GPU 2，以此类推。< font color='red'>数据像流水线一样从第一张卡传到最后一张卡。</font>
- 代码中的体现：pipeline_model_parallel_size 决定了模型被切分成几段。

#### 3. 虚拟流水线并行 (Virtual Pipeline Parallelism, VPP)
- 对应参数：<font color='red'>virtual</font>_pipeline_model_parallel_size
- 通俗解释：“让流水线跑得更快”。
    - 在普通的 PP 中，GPU 经常需要等待上一张卡传数据（产生气泡）。VPP 允许一张 GPU 同时负责模型的好几小段（微批次），通过交错执行来大幅减少等待时间，提高 GPU 利用率。
#### 4. 专家并行 (Expert Parallelism, EP) 与 专家张量并行 (ETP)
- 对应参数：<font color='red'>expert</font>_model_parallel_size 和 <font color='red'>expert</font>_tensor_parallel_size
- 通俗解释：这是专门为 MoE（混合专家模型） 设计的。MoE 模型中有许多“专家”网络，<font color='red'>EP 负责把这些不同的专家分散到不同的 GPU 上</font>；而 ETP 则是在单个专家内部再进行张量切分。这能极大地扩展模型的参数量（如 DeepSeek-V3、GPT-4 等架构）。

#### 5. 上下文并行 (Context Parallelism, CP)
- 对应参数：<font color='red'>context</font>_parallel_size
- 通俗解释：“把超长文本切开算”。当输入序列极长（如 100K 上下文）时，单卡存不下整个序列的注意力矩阵（KV Cache）。CP 会把序列拆分到多张卡上计算，再通过通信交换必要的信息。

### ⚙️ 其他辅助参数
- <font color='red'>pipeline_model_parallel_split_rank</font>：指定流水线在哪个位置切分（通常<font color='green'>用于 Encoder-Decoder 架构，区分编码器和解码器的流水线切分点</font>）。
    - 它专门用于 Encoder-Decoder（编码器-解码器）架构 的模型（例如 T5、BART 或早期的 mBART 等）
    - <font color='red'>在纯 Decoder 架构（如 GPT、Llama、Qwen）的模型中，这个参数通常设置为 None 或 0，但在处理 Encoder-Decoder 架构时，它的作用至关重要。</font>
    
- <font color='red'>use_sharp</font>：是否使用 NVIDIA SHARP（Scalable Hierarchical Aggregation and Reduction Protocol）技术来加速跨节点的集合通信。

- <font color='red'>nccl_communicator_config_path</font>：<font color='green'>指定底层 NCCL 通信库的配置文件路径，用于微调跨节点通信的性能。</font>

### 📌 总结：它到底在做什么？
当这行代码执行完毕后，Megatron 并没有开始真正的计算，而是完成了一张“通信地图”的绘制：
1. 分组：它根据总 GPU 数量和你传入的各个并行度（TP, PP, DP, EP...），<font color='green'>计算出每张 GPU 到底属于哪个组。</font>
    - 例如，<font color='red'>GPU 0 可能同时属于“TP组[0,1]”、“PP组[0,4,8]”和“DP组[0,2]”</font>。
2. 建群：它在底层调用 PyTorch 的 <font color='green'>torch.distributed.new_group()，为上述每一个组建立专属的通信通道。</font>
3. 准备：后续模型在计算时，只要说“我要在 TP 组内同步梯度”，框架就能立刻找到对应的通信通道并执行高效的 all-reduce 操作。

简单来说，<font color='red'>这是 VeRL 框架在启动训练前，调用 Megatron 为大规模模型搭建的底层分布式通信骨架。</font>



# torch.distributed.new_group()
- torch.distributed.new_group 是 PyTorch 分布式通信模块（torch.distributed）中的一个核心接口，<font color='red'>主要用于创建自定义的进程组（Process Group）。</font>

- <font color='red'>在分布式训练中，默认情况下所有进程（即整个 world）都在同一个全局通信组内。</font><font color='green'>但在一些复杂的并行策略（如大模型训练中的张量并行、流水线并行）或特定的通信需求下，我们需要让一部分进程进行细粒度的通信，这时就需要用到 new_group</font>

## 🛠️ 核心功能与参数
它的基本用法是通过指定一部分进程的 rank（进程编号）来<font color='red'>创建一个子组。</font>

In [ ]:
import torch.distributed as dist

# 创建一个包含 rank 0 和 rank 1 的通信组
ranks = [0, 1]
group = dist.new_group(ranks, backend='nccl')

### 主要参数说明：
- ranks：一个整数列表，<font color='red'>指定哪些进程（rank）属于这个新的通信组。</font>
- backend：指定该组使用的通信后端，如 'nccl'（推荐用于 GPU 通信）或 'gloo'（常用于 CPU 通信）。<font color='red'>如果不指定，默认使用全局进程组的后端。</font>
- timeout：组内操作（如集合通信）的超时时间，默认通常是 30 分钟。

## 💡 核心作用
### 1.细粒度的通信控制：
- 在使用集合通信算子（如 all_reduce, all_gather, barrier 等）时，可以通过 group 参数指定通信范围。
- <font color='red'>这样，数据交换和同步就只会发生在该组内的 rank 之间，而不是等待所有进程。</font>

In [ ]:
# 只有指定 group 内的 rank 会在这里进行同步
dist.all_reduce(tensor, group=group)
dist.barrier(group=group)

### 2.点对点通信（P2P）的标识符：
- 如果两个 rank 之间需要同时进行多组不同的点对点通信（send / recv），不同的 group 可以作为唯一的通信标识符，用来区分不同的通信流，避免数据错乱。

### 3.混合后端通信：
- <font color='red'>你可以创建多个通信组，并指定不同的后端。</font>例如，:
    - 某些组使用 nccl 进行高效的 GPU 张量通信，
    - 而另一些组使用 gloo 进行 CPU 张量通信。
    
## ⚠️ 关键规则与常见误区（非常重要）
在使用 new_group 时，新手最容易踩的坑就是死锁（Deadlock）或 NCCL 错误。PyTorch 对创建通信组有两个强制要求：

- 全局参与：所有参与了全局初始化的进程（即整个 world 中的所有 rank），都必须执行 new_group 这行代码，即使某些进程并不属于这个新创建的组。
- 顺序一致：所有进程创建通信组的顺序必须完全相同。

## ❌ 错误的写法（会导致死锁或报错）：
- 原因： 创建子组时，底层（如 NCCL）需要生成一个唯一的通信 ID，并在所有进程间进行同步。如果 rank 2 和 3 跳过了创建步骤，通信 ID 就无法达成一致，导致程序卡死或崩溃。

In [ ]:
# 错误示例：只有组内成员才执行 new_group
if rank in [0, 1]:
    subgroup = dist.new_group([0, 1])

## ✅ 正确的写法：

In [ ]:
group = None
rank = dist.get_rank()

# 所有进程都必须按相同的顺序执行这段代码
ranks_in_group = [0, 1]
subgroup = dist.new_group(ranks_in_group)

# 只有组内成员才保留该组的句柄，用于后续通信
if rank in ranks_in_group:
    group = subgroup

### 如果你需要创建多个通信组，也必须保证所有进程按照完全相同的顺序依次创建：
- 通过这种机制，PyTorch 能够确保分布式环境下的通信拓扑被正确、安全地建立起来。

In [ ]:
group1 = dist.new_group([0, 1])
group2 = dist.new_group([2, 3])
group3 = dist.new_group([1, 2, 3])
# 所有进程都会按顺序创建 group1, group2, group3
# 然后各自判断自己属于哪个组，并保存对应的句柄